## Notebook 03 — Streamlit Dashboard (Implementation + Logic)

**Goal:** Provide a clean, reproducible Streamlit dashboard that loads the artifacts generated by Notebook 01 & 02 and supports:
- EDA snapshots (hourly surge pattern)
- Spatial visualization (interactive Folium heatmap)
- Live prediction (surge probability + multiplier)

Artifacts expected:
- `outputs/trips_processed.parquet`
- `outputs/models/clf_best.joblib`
- `outputs/models/reg_best.joblib`

Run:
```bash
streamlit run app/app.py
```


**Artifact verification:** Before running the dashboard, ensure all outputs from Notebooks 01 and 02 exist.

In [1]:
from __future__ import annotations

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

paths = {
    "processed_parquet": PROJECT_ROOT / "outputs" / "trips_processed.parquet",
    "clf_model": PROJECT_ROOT / "outputs" / "models" / "clf_best.joblib",
    "reg_model": PROJECT_ROOT / "outputs" / "models" / "reg_best.joblib",
    "streamlit_app": PROJECT_ROOT / "app" / "app.py",
}

for k, p in paths.items():
    print(f"{k}: {p} -> {'OK' if p.exists() else 'MISSING'}")


processed_parquet: c:\Users\blase\OneDrive\Documents\last_DataScienceProject\outputs\trips_processed.parquet -> OK
clf_model: c:\Users\blase\OneDrive\Documents\last_DataScienceProject\outputs\models\clf_best.joblib -> OK
reg_model: c:\Users\blase\OneDrive\Documents\last_DataScienceProject\outputs\models\reg_best.joblib -> OK
streamlit_app: c:\Users\blase\OneDrive\Documents\last_DataScienceProject\app\app.py -> OK


**Interpretation:** All paths OK means the dashboard can load data and models. Any MISSING path requires re-running the corresponding notebook (01 for parquet, 02 for models).

### Dashboard architecture

The Streamlit app (`app/app.py`) follows a simple pattern:
- **Load artifacts** using Streamlit caching
- **Render EDA** (fast aggregates / charts)
- **Render maps** (Folium HTML embedded)
- **Collect user inputs** (time, coords, distance/time)
- **Featurize input** using the same feature engineering as training (`preprocess_trips_df(..., add_target=False)`)
- **Predict** using the saved pipelines (`clf_best.joblib`, `reg_best.joblib`)

This ensures the app uses *exactly the same feature transformations* as the notebook pipeline.


### Feature set used by the model and dashboard

The Surge Prediction Model uses: **trip features** (distance, duration, speed, haversine, bearing, distance-vs-haversine ratio), **temporal features** (hour, day of week, cyclical encodings, peak flags), **spatial/IDs** (service_type_id), and **weather: rain_flag only** (1 if precipitation > 0, 0 otherwise).

The dashboard passes the same features. For the prediction panel, the user selects **"Raining? Yes/No"**, which sets `rain_flag`; the app does not fetch live weather. This keeps the report and code in sync: the model is trained with `rain_flag` from the merged weather in Notebook 01, and at inference the dashboard provides `rain_flag` from user input. Classification uses **surge_event** (threshold 0.4) and regression uses **log1p(multiplier)** with expm1 for display (Notebook 02).